In [1]:
import numpy as np
import torch 
import pyro
import re
import pandas as pd

import importlib
import time
import pickle
import os

from granch_utils import init_model_tensor, init_stimuli_tensor,init_params_tensor, main_sim_tensor, lesioned_sim, compute_prob_tensor,  num_stab_help, proxy_sim, get_embedding, get_sequence
#importlib.reload(granch_utils)
importlib.reload(num_stab_help)
importlib.reload(init_model_tensor)
importlib.reload(init_stimuli_tensor)
importlib.reload(main_sim_tensor)
importlib.reload(lesioned_sim)
importlib.reload(get_embedding)
importlib.reload(get_sequence)



<module 'granch_utils.get_sequence' from '/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/get_sequence.py'>

# Main TEST

In [8]:
importlib.reload(main_sim_tensor)
importlib.reload(lesioned_sim)
importlib.reload(init_params_tensor)
importlib.reload(init_model_tensor)
importlib.reload(compute_prob_tensor)
importlib.reload(proxy_sim)



summarized_main_sim  = []

 #trial_info = pd.read_csv("/om2/scratch/tmp/galraz/RANCH/RANCH_cluster/sim_info/trial_info/trial_info_graded_dishab.csv")\
trial_info = pd.read_csv("/Users/caoanjie/Desktop/projects/RANCH/RANCH_cluster/sim_info/trial_info/stimulus_type/adults/trial_info.csv")

    # just first row
    #trial_info = trial_info.iloc[2]
    #trial_info = trial_info.to_dict()

background_trials = trial_info[trial_info["violation_type"]=="background"]
identity_trials = trial_info[trial_info["violation_type"]=="identity"]
number_trials = trial_info[trial_info["violation_type"]=="number"]
pose_trials = trial_info[trial_info["violation_type"]=="pose"]
animacy_trials = trial_info[trial_info["violation_type"]=="animacy"]


background_trial_info = background_trials.iloc[np.random.randint(0, len(background_trials))].to_dict()
identity_trial_info = identity_trials.iloc[np.random.randint(0, len(identity_trials))].to_dict()
number_trial_info = number_trials.iloc[np.random.randint(0, len(number_trials))].to_dict()
pose_trial_info = pose_trials.iloc[np.random.randint(0, len(pose_trials))].to_dict()
animacy_trial_info = animacy_trials.iloc[np.random.randint(0, len(animacy_trials))].to_dict()

    # 2. Convert Stimuli_info into actual embedding 
    #fam, test = get_embedding.string_to_embedding(trial_info = trial_info)
#background_fam, background_test = get_embedding.string_to_embedding(trial_info = background_trial_info)
#identity_fam, identity_test = get_embedding.string_to_embedding(trial_info = identity_trial_info)
#num_fam, num_test = get_embedding.string_to_embedding(trial_info = number_trial_info)
#pose_fam, pose_test = get_embedding.string_to_embedding(trial_info = pose_trial_info)
#animacy_fam, animacy_test = get_embedding.string_to_embedding(trial_info = animacy_trial_info)

background_fam, background_test = torch.tensor([0.4533, -0.9634, -0.7922]), torch.tensor([0.4533, -0.9634, -0.7922])
identity_fam, identity_test = torch.tensor([ 0.8621, -0.0764,  0.2498]), torch.tensor([1.5216,  0.1411, -0.1434])
num_fam, num_test = torch.tensor([ -0.3219,  0.8121,  0.7056]), torch.tensor([-0.5816,  1.4219,  0.2670])
pose_fam, pose_test = torch.tensor([ 1.8067, -0.2556, -0.6386]), torch.tensor([1.6578, -0.1109, -0.7233])
animacy_fam, animacy_test = torch.tensor([-1.3875, -1.4487,  0.0324]), torch.tensor([1.8204, -0.4493, -0.0524])
    
    

    #print("fam:", fam, "test:", test)


    # 3. Convert trial information into sequence
    #sequence_scheme = get_sequence.param_to_scheme(trial_info=trial_info)

sequence_scheme = "BBBBBD"
    #background_sequence_scheme = get_sequence.param_to_scheme(trial_info=background_trial_info)
    #identity_sequence_scheme = get_sequence.param_to_scheme(trial_info=identity_trial_info)
    #number_sequence_scheme = get_sequence.param_to_scheme(trial_info=number_trial_info)
    #pose_sequence_scheme = get_sequence.param_to_scheme(trial_info=pose_trial_info)
    #animacy_sequence_scheme = get_sequence.param_to_scheme(trial_info=animacy_trial_info)
background_sequence_scheme = sequence_scheme
identity_sequence_scheme = sequence_scheme
number_sequence_scheme = sequence_scheme
pose_sequence_scheme = sequence_scheme
animacy_sequence_scheme = sequence_scheme
    # ------ Set up simulation raw material ------ #
    # 4. Set up stimuli
background_s = init_stimuli_tensor.granch_stimuli(background_trial_info["feature_n"], background_sequence_scheme)
background_s.add_stimuli_sequence(background_fam, background_test)

identity_s = init_stimuli_tensor.granch_stimuli(identity_trial_info["feature_n"], identity_sequence_scheme)
identity_s.add_stimuli_sequence(identity_fam, identity_test)

number_s = init_stimuli_tensor.granch_stimuli(number_trial_info["feature_n"], number_sequence_scheme)
number_s.add_stimuli_sequence(num_fam, num_test)

pose_s = init_stimuli_tensor.granch_stimuli(pose_trial_info["feature_n"], pose_sequence_scheme)
pose_s.add_stimuli_sequence(pose_fam, pose_test)

animacy_s = init_stimuli_tensor.granch_stimuli(animacy_trial_info["feature_n"], animacy_sequence_scheme)
animacy_s.add_stimuli_sequence(animacy_fam, animacy_test)



for i in range(100): 
    print(i)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    BATCH_INFO = {
        "jitter_n": 1, 
        "total_batch_n": 1, 
        "jitter_mode": "sampling"
    }

    GRID_INFO = {
        "grid_mu_start": -4, "grid_mu_end": 4, "grid_mu_step": 5, 
        "grid_sigma_start": 0.001, "grid_sigma_end": 1.8, "grid_sigma_step": 5, 
        "grid_y_start": -4, "grid_y_end": 4, "grid_y_step": 5, 
        "grid_epsilon_start": 0.00000000001, "grid_epsilon_end": 1, "grid_epsilon_step": 5, 
        "hypothetical_obs_grid_n": 10
    }


    BATCH_GRID_INFO = num_stab_help.get_batch_grid(BATCH_INFO, GRID_INFO)

    PRIOR_INFO = {
        "mu_prior": 0,  
        "V_prior":3, 
        "alpha_prior": 10, 
        "beta_prior": 0.1, 
        "epsilon": 0.0001, "mu_epsilon":0.001	, "sd_epsilon": 0.1, 
        "hypothetical_obs_grid_n": 5, 
        "world_EIGs": 0.0001	, "max_observation": 500
    }

    # tensor_stimuli = num_stab_help.sample_spore_experiment(pair_each_stim = 1, n_feature=3)

   
    #tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], s)
    
    background_tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], background_s)
    identity_tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], identity_s)
    number_tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], number_s)
    pose_tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], pose_s)
    animacy_tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], animacy_s)



    params = init_params_tensor.granch_params(
                    grid_mu =  BATCH_GRID_INFO["grid_mus"][0].to(device),
                    grid_sigma = BATCH_GRID_INFO["grid_sigmas"][0].to(device),
                    grid_y = BATCH_GRID_INFO["grid_ys"][0].to(device),
                    grid_epsilon = BATCH_GRID_INFO["grid_epsilons"][0].to(device),
                    hypothetical_obs_grid_n = PRIOR_INFO["hypothetical_obs_grid_n"], 
                    mu_prior = PRIOR_INFO["mu_prior"],
                    V_prior = PRIOR_INFO["V_prior"], 
                    alpha_prior = PRIOR_INFO["alpha_prior"], 
                    beta_prior = PRIOR_INFO["beta_prior"],
                    epsilon  = PRIOR_INFO["epsilon"], 
                    mu_epsilon = PRIOR_INFO["mu_epsilon"], 
                    sd_epsilon = PRIOR_INFO["sd_epsilon"], 
                    world_EIGs = PRIOR_INFO["world_EIGs"],
                    max_observation = PRIOR_INFO["max_observation"], 
                    forced_exposure_max= np.nan, 
                    linking_hypothesis = "EIG")
            
                # add the various different cached bits
    params.add_meshed_grid()
    params.add_lp_mu_sigma()
    params.add_y_given_mu_sigma()
    params.add_lp_epsilon()
    params.add_priors()
    pd.set_option('display.max_rows', None)


    if params.linking_hypothesis == "EIG": 
        background_res = main_sim_tensor.granch_main_simulation(params, background_tensor_model, background_s).output
        identity_res = main_sim_tensor.granch_main_simulation(params, identity_tensor_model, identity_s).output
        number_res = main_sim_tensor.granch_main_simulation(params, number_tensor_model, number_s).output
        pose_res = main_sim_tensor.granch_main_simulation(params, pose_tensor_model, pose_s).output
        animacy_res = main_sim_tensor.granch_main_simulation(params, animacy_tensor_model, animacy_s).output

        #lesioned_res = lesioned_sim.granch_no_noise_simulation(params, tensor_model, s)

        background_res["run_id"] = i
        identity_res["run_id"] = i
        number_res["run_id"] = i
        pose_res["run_id"] = i
        animacy_res["run_id"] = i

        background_res["v_type"] = "background"
        identity_res["v_type"] = "identity"
        number_res["v_type"] = "number"
        pose_res["v_type"] = "pose"
        animacy_res["v_type"] = "animacy"

        summarized_main_sim.append(background_res)
        summarized_main_sim.append(identity_res)
        summarized_main_sim.append(number_res)
        summarized_main_sim.append(pose_res)
        summarized_main_sim.append(animacy_res)
        print(summarized_main_sim)

        
    else: 
        #print(params.linking_hypothesis)
        res = proxy_sim.granch_proxy_sim(params, tensor_model, s)
        output = res.output

        output["run_id"] = i
        print(i)
        summarized_main_sim.append(output)

   

main_sims_df = pd.concat(summarized_main_sim)


#res = lesioned_sim.granch_no_learning_simulation(params, tensor_model, tensor_stimuli[0])
#res = lesioned_sim.granch_no_noise_simulation(params, tensor_model, tensor_stimuli[0])


#res.behavior.to_csv("diagnostics/s.csv")

main_sims_df.to_csv("stim_type_replicate.csv")

0


/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.45337846875190735' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '-0.9635217189788818' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and 

[   sample_n  run_id      v_type
0        46       0  background
1        14       0  background
2         8       0  background
3         1       0  background
4         2       0  background
5         8       0  background,    sample_n  run_id    v_type
0        26       0  identity
1         9       0  identity
2         4       0  identity
3         1       0  identity
4         2       0  identity
5         4       0  identity,    sample_n  run_id  v_type
0        10       0  number
1        22       0  number
2        19       0  number
3        34       0  number
4         1       0  number
5        35       0  number,    sample_n  run_id v_type
0        51       0   pose
1        10       0   pose
2         4       0   pose
3         7       0   pose
4        11       0   pose
5         3       0   pose,    sample_n  run_id   v_type
0        70       0  animacy
1         2       0  animacy
2        43       0  animacy
3        30       0  animacy
4        12       0  animacy
5 

/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.4532662332057953' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '-0.9633610844612122' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and w

[   sample_n  run_id      v_type
0        46       0  background
1        14       0  background
2         8       0  background
3         1       0  background
4         2       0  background
5         8       0  background,    sample_n  run_id    v_type
0        26       0  identity
1         9       0  identity
2         4       0  identity
3         1       0  identity
4         2       0  identity
5         4       0  identity,    sample_n  run_id  v_type
0        10       0  number
1        22       0  number
2        19       0  number
3        34       0  number
4         1       0  number
5        35       0  number,    sample_n  run_id v_type
0        51       0   pose
1        10       0   pose
2         4       0   pose
3         7       0   pose
4        11       0   pose
5         3       0   pose,    sample_n  run_id   v_type
0        70       0  animacy
1         2       0  animacy
2        43       0  animacy
3        30       0  animacy
4        12       0  animacy
5 

...

[   sample_n  run_id      v_type
0        46       0  background
1        14       0  background
2         8       0  background
3         1       0  background
4         2       0  background
5         8       0  background,    sample_n  run_id    v_type
0        26       0  identity
1         9       0  identity
2         4       0  identity
3         1       0  identity
4         2       0  identity
5         4       0  identity,    sample_n  run_id  v_type
0        10       0  number
1        22       0  number
2        19       0  number
3        34       0  number
4         1       0  number
5        35       0  number,    sample_n  run_id v_type
0        51       0   pose
1        10       0   pose
2         4       0   pose
3         7       0   pose
4        11       0   pose
5         3       0   pose,    sample_n  run_id   v_type
0        70       0  animacy
1         2       0  animacy
2        43       0  animacy
3        30       0  animacy
4        12       0  animacy
5 

/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.453278124332428' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '-0.9631282091140747' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and wi

[   sample_n  run_id      v_type
0        46       0  background
1        14       0  background
2         8       0  background
3         1       0  background
4         2       0  background
5         8       0  background,    sample_n  run_id    v_type
0        26       0  identity
1         9       0  identity
2         4       0  identity
3         1       0  identity
4         2       0  identity
5         4       0  identity,    sample_n  run_id  v_type
0        10       0  number
1        22       0  number
2        19       0  number
3        34       0  number
4         1       0  number
5        35       0  number,    sample_n  run_id v_type
0        51       0   pose
1        10       0   pose
2         4       0   pose
3         7       0   pose
4        11       0   pose
5         3       0   pose,    sample_n  run_id   v_type
0        70       0  animacy
1         2       0  animacy
2        43       0  animacy
3        30       0  animacy
4        12       0  animacy
5 

/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.4533509314060211' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '-0.9634145498275757' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.all_observations.loc[self.current_t]  = Normal(current_stimulus, noise_epsilon).sample().tolist()
/Users/caoanjie/Desktop/projects/RANCH/RANCH_model/granch_utils/init_model_tensor.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and w

[   sample_n  run_id      v_type
0        46       0  background
1        14       0  background
2         8       0  background
3         1       0  background
4         2       0  background
5         8       0  background,    sample_n  run_id    v_type
0        26       0  identity
1         9       0  identity
2         4       0  identity
3         1       0  identity
4         2       0  identity
5         4       0  identity,    sample_n  run_id  v_type
0        10       0  number
1        22       0  number
2        19       0  number
3        34       0  number
4         1       0  number
5        35       0  number,    sample_n  run_id v_type
0        51       0   pose
1        10       0   pose
2         4       0   pose
3         7       0   pose
4        11       0   pose
5         3       0   pose,    sample_n  run_id   v_type
0        70       0  animacy
1         2       0  animacy
2        43       0  animacy
3        30       0  animacy
4        12       0  animacy
5 

In [6]:
print(background_fam, background_test)
print(identity_fam, identity_test)
print(num_fam, num_test)
print(pose_fam, pose_test)
print(animacy_fam, animacy_test)

tensor([ 0.4533, -0.9634, -0.7922], dtype=torch.float64) tensor([ 0.4533, -0.9634, -0.7922], dtype=torch.float64)
tensor([ 0.8621, -0.0764,  0.2498], dtype=torch.float64) tensor([ 1.5216,  0.1411, -0.1434], dtype=torch.float64)
tensor([-0.3219,  0.8121,  0.7056], dtype=torch.float64) tensor([-0.5816,  1.4219,  0.2670], dtype=torch.float64)
tensor([ 1.8067, -0.2556, -0.6386], dtype=torch.float64) tensor([ 1.6578, -0.1109, -0.7233], dtype=torch.float64)
tensor([-1.3875, -1.4487,  0.0324], dtype=torch.float64) tensor([ 1.8204, -0.4493, -0.0524], dtype=torch.float64)


In [37]:
summarized_main_sim  = []
eig_dfs = []


for i in range(400): 
    res = main_sim_tensor.granch_main_simulation(params, tensor_model, s)

    output = res.output
    eig_df = res.behavior

    output["run_id"] = i
    eig_df["run_id"] = i

    summarized_main_sim.append(output)
    eig_dfs.append(eig_df)

main_sims_df = pd.concat(summarized_main_sim)
main_eig_dfs = pd.concat(eig_dfs)

#main_sims_df.to_csv("x1000_parameters.csv")
main_sims_df.to_csv("mu0_x0.01_parameters.csv")

# Specific Grid Test

In [548]:
importlib.reload(main_sim_tensor)
importlib.reload(lesioned_sim)
importlib.reload(init_params_tensor)
importlib.reload(init_model_tensor)
importlib.reload(compute_prob_tensor)
importlib.reload(proxy_sim)
from torch.distributions import Normal, uniform


# test stimuli
test_stimuli = init_model_tensor.granch_stimuli(3, "BBBBBB")
test_stimuli.add_toy_example(0.3, 0.7)

# test grid  
grid_mu_distribution = uniform.Uniform(.2, .8)
grid_sigma_distribution = uniform.Uniform(.2, .8)
grid_y_distribution = uniform.Uniform(.2, .8)
grid_epsilon_distribution = uniform.Uniform(.2, .8)

# grid_mu = grid_mu_distribution.sample([2, ])
# grid_sigma = grid_sigma_distribution.sample([2, ])
# grid_y = grid_y_distribution.sample([2, ])
# grid_epsilon = grid_epsilon_distribution.sample([2, ])

grid_mu = torch.linspace(start = .2, end  = .8, steps = 5)
grid_sigma = torch.linspace(start = .2, end  = .8, steps = 5)
grid_y = torch.linspace(start = .2, end  = .8, steps = 5)
grid_epsilon = torch.linspace(start = .2, end  = .8, steps = 5)

# test prior
PRIOR_INFO = {
    "mu_prior": 1,  
    "V_prior": 1, 
    "alpha_prior": 1, 
    "beta_prior": 1, 
    "epsilon": 0.0001, "mu_epsilon":0.0001, "sd_epsilon": 0.0001, 
    "hypothetical_obs_grid_n": 2, 
    "world_EIGs": 0.001, "max_observation": 500
}


tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], test_stimuli)

params = init_params_tensor.granch_params(
                grid_mu =  grid_mu,
                grid_sigma = grid_sigma,
                grid_y = grid_y, 
                grid_epsilon = grid_epsilon,
                hypothetical_obs_grid_n = 5, 
                mu_prior = PRIOR_INFO["mu_prior"],
                V_prior = PRIOR_INFO["V_prior"], 
                alpha_prior = PRIOR_INFO["alpha_prior"], 
                beta_prior = PRIOR_INFO["beta_prior"],
                epsilon  = PRIOR_INFO["epsilon"], 
                mu_epsilon = PRIOR_INFO["mu_epsilon"], 
                sd_epsilon = PRIOR_INFO["sd_epsilon"], 
                world_EIGs = PRIOR_INFO["world_EIGs"],
                max_observation = PRIOR_INFO["max_observation"], 
                forced_exposure_max= np.nan, 
                linking_hypothesis = "EIG")
        
            # add the various different cached bits
params.add_meshed_grid()
params.add_lp_mu_sigma()
params.add_y_given_mu_sigma()
params.add_lp_epsilon()
params.add_priors()

if params.linking_hypothesis == "EIG": 
    res = main_sim_tensor.granch_main_simulation(params, tensor_model, test_stimuli)
else: 
    res = proxy_sim.granch_proxy_sim(params, tensor_model, test_stimuli)

res.behavior


/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30011168122291565' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.29988840222358704' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.2999730110168457' has dtype incompatible with int64, please explicitly cast to a compatible d

tensor(-12.4852, dtype=torch.float64)
tensor(-13.2946, dtype=torch.float64)
tensor(-13.3100, dtype=torch.float64)
tensor(-13.3129, dtype=torch.float64)
tensor(-13.3140, dtype=torch.float64)
tensor(-13.3179, dtype=torch.float64)
tensor(-13.3183, dtype=torch.float64)
tensor(-13.3217, dtype=torch.float64)
tensor(-13.3198, dtype=torch.float64)
tensor(-13.3197, dtype=torch.float64)
tensor(-13.3204, dtype=torch.float64)
tensor(-13.3200, dtype=torch.float64)
tensor(-13.3205, dtype=torch.float64)
tensor(-13.3215, dtype=torch.float64)
tensor(-13.3208, dtype=torch.float64)
tensor(-13.3238, dtype=torch.float64)
tensor(-13.3208, dtype=torch.float64)
tensor(-13.3202, dtype=torch.float64)
tensor(-13.3217, dtype=torch.float64)
tensor(-13.3218, dtype=torch.float64)
tensor(-13.3205, dtype=torch.float64)
tensor(-13.3230, dtype=torch.float64)
tensor(-13.3198, dtype=torch.float64)
tensor(-13.3220, dtype=torch.float64)
tensor(-13.3225, dtype=torch.float64)
tensor(-13.3235, dtype=torch.float64)
tensor(-13.3

,stimulus_id,EIG,Look_away,surprisal,kl
0,0,NaN,False,-12.485166,1.434026e-01
1,0,NaN,False,-13.294644,8.591513e-04
2,0,NaN,False,-13.309959,1.427475e-04
3,0,NaN,False,-13.312906,3.384517e-05
4,0,NaN,False,-13.314026,7.788265e-06
5,0,NaN,False,-13.317922,1.050467e-06
6,0,NaN,False,-13.318296,2.218626e-07
7,0,NaN,False,-13.321696,1.064257e-06
8,0,NaN,False,-13.319807,2.046861e-06
9,0,NaN,False,-13.319726,2.779853e-06


# Visualizing random grid KL (500)

In [544]:
importlib.reload(main_sim_tensor)
importlib.reload(lesioned_sim)
importlib.reload(init_params_tensor)
importlib.reload(compute_prob_tensor)
importlib.reload(proxy_sim)


def run_random_grid_and_visualize_kl():


    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    BATCH_INFO = {
        "jitter_n": 1, 
        "total_batch_n": 1, 
        "jitter_mode": "sampling"
    }

    GRID_INFO = {
       "grid_mu_start": -4, "grid_mu_end": 4, "grid_mu_step": 5, 
       "grid_sigma_start": 0.001, "grid_sigma_end": 1.8, "grid_sigma_step": 5, 
       "grid_y_start": -4, "grid_y_end": 4, "grid_y_step": 5, 
       "grid_epsilon_start": 0.000001, "grid_epsilon_end": 1, "grid_epsilon_step": 5, 
       "hypothetical_obs_grid_n": 10
    }


#    GRID_INFO = {
#        "grid_mu_start": .2, "grid_mu_end": .8, "grid_mu_step": 5, 
#        "grid_sigma_start": .2, "grid_sigma_end": .8, "grid_sigma_step": 5, 
#        "grid_y_start":.2, "grid_y_end": 8, "grid_y_step": 5, 
#        "grid_epsilon_start": .2, "grid_epsilon_end": .8, "grid_epsilon_step": 5, 
#        "hypothetical_obs_grid_n": 10
#    }



    BATCH_GRID_INFO = num_stab_help.get_batch_grid(BATCH_INFO, GRID_INFO)

    PRIOR_INFO = {
        "mu_prior": 1,  
        "V_prior": 1, 
        "alpha_prior": 3, 
        "beta_prior": 1, 
        "epsilon": 0.0001, "mu_epsilon":0.0001, "sd_epsilon": 0.0001, 
        "hypothetical_obs_grid_n": 2, 
        "world_EIGs": 0.01, "max_observation": 500
    }


    test_stimuli = init_model_tensor.granch_stimuli(3, "BBBBBD")
    test_stimuli.add_toy_example(0.3, 0.7)
    



    tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], test_stimuli)

    params = init_params_tensor.granch_params(
                    grid_mu =  BATCH_GRID_INFO["grid_mus"][0].to(device),
                    grid_sigma = BATCH_GRID_INFO["grid_sigmas"][0].to(device),
                    grid_y = BATCH_GRID_INFO["grid_ys"][0].to(device),
                    grid_epsilon = BATCH_GRID_INFO["grid_epsilons"][0].to(device),
                    hypothetical_obs_grid_n = PRIOR_INFO["hypothetical_obs_grid_n"], 
                    mu_prior = PRIOR_INFO["mu_prior"],
                    V_prior = PRIOR_INFO["V_prior"], 
                    alpha_prior = PRIOR_INFO["alpha_prior"], 
                    beta_prior = PRIOR_INFO["beta_prior"],
                    epsilon  = PRIOR_INFO["epsilon"], 
                    mu_epsilon = PRIOR_INFO["mu_epsilon"], 
                    sd_epsilon = PRIOR_INFO["sd_epsilon"], 
                    world_EIGs = PRIOR_INFO["world_EIGs"],
                    max_observation = PRIOR_INFO["max_observation"], 
                    forced_exposure_max= np.nan, 
                    linking_hypothesis = "surprisal")
            
                # add the various different cached bits
    params.add_meshed_grid()
    params.add_lp_mu_sigma()
    params.add_y_given_mu_sigma()
    params.add_lp_epsilon()
    params.add_priors()

    if params.linking_hypothesis == "EIG": 
        res = main_sim_tensor.granch_main_simulation(params, tensor_model, test_stimuli)
    else: 
        res = proxy_sim.granch_proxy_sim(params, tensor_model, test_stimuli)

    return res


#res = lesioned_sim.granch_no_learning_simulation(params, tensor_model, tensor_stimuli[0])
#res = lesioned_sim.granch_no_noise_simulation(params, tensor_model, tensor_stimuli[0])



# try it out 5 eims
res_list = []
#kl_df_list = []
for i in range(2000): 
    print(i)
    res= (run_random_grid_and_visualize_kl()).behavior
    res["id"] = i
    res_list.append(res)
    #kl_df_list.append(kl_df)

main_res = pd.concat(res_list)
#kl_df_res = pd.concat(kl_df_list)

main_res.to_csv("diagnostics/sim_res.csv")
#main_res.dropna()
#print(res.behavior)
#main_sim_tensor.granch_main_simulation(PRIOR_INFO, tensor_model, tensor_stimuli)



0
tensor(-4.3281e-08, dtype=torch.float64)
tensor(-4.3281e-08, dtype=torch.float64)
tensor(-4.3281e-08, dtype=torch.float64)
tensor(-4.3281e-08, dtype=torch.float64)
tensor(-4.3281e-08, dtype=torch.float64)
tensor(-4.3281e-08, dtype=torch.float64)
1
tensor(-2.0384, dtype=torch.float64)
tensor(-2.4123, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4123, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4119, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.29983359575271606' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30007535219192505' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.3001207709312439' has dtype incompatible with int64, please explicitly cast to a compatible d

tensor(-2.4123, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4124, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4123, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4120, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4121, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
tensor(-2.4122, dtype=torch.float64)
t

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30001121759414673' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30006733536720276' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.29989752173423767' has dtype incompatible with int64, please explicitly cast to a compatible 

tensor(-8.1743, dtype=torch.float64)
tensor(-8.1819, dtype=torch.float64)
tensor(-8.1764, dtype=torch.float64)
tensor(-8.1757, dtype=torch.float64)
tensor(-8.1745, dtype=torch.float64)
tensor(-8.1786, dtype=torch.float64)
tensor(-8.1840, dtype=torch.float64)
tensor(-8.1849, dtype=torch.float64)
tensor(-8.1729, dtype=torch.float64)
tensor(-8.1730, dtype=torch.float64)
tensor(-8.1743, dtype=torch.float64)
tensor(-8.1754, dtype=torch.float64)
tensor(-8.1859, dtype=torch.float64)
tensor(-8.1829, dtype=torch.float64)
tensor(-8.1746, dtype=torch.float64)
tensor(-8.1708, dtype=torch.float64)
tensor(-8.1719, dtype=torch.float64)
tensor(-8.1770, dtype=torch.float64)
tensor(-8.1740, dtype=torch.float64)
tensor(-8.1790, dtype=torch.float64)
tensor(-8.1699, dtype=torch.float64)
tensor(-8.1716, dtype=torch.float64)
tensor(-8.1686, dtype=torch.float64)
tensor(-8.1730, dtype=torch.float64)
tensor(-8.1767, dtype=torch.float64)
tensor(-8.1738, dtype=torch.float64)
tensor(-8.1680, dtype=torch.float64)
t

...

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30005332827568054' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.30007392168045044' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.3000729978084564' has dtype incompatible with int64, please explicitly cast to a compatible d

tensor(-0.6492, dtype=torch.float64)
tensor(-0.6490, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6488, dtype=torch.float64)
tensor(-0.6493, dtype=torch.float64)
tensor(-0.6491, dtype=torch.float64)
tensor(-0.6493, dtype=torch.float64)
tensor(-0.6489, dtype=torch.float64)
tensor(-0.6494, dtype=torch.float64)
tensor(-0.6494, dtype=torch.float64)
tensor(-0.6490, dtype=torch.float64)
tensor(-0.6497, dtype=torch.float64)
tensor(-0.6488, dtype=torch.float64)
tensor(-0.6491, dtype=torch.float64)
tensor(-0.6493, dtype=torch.float64)
tensor(-0.6491, dtype=torch.float64)
tensor(-0.6489, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6494, dtype=torch.float64)
tensor(-0.6491, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6494, dtype=torch.float64)
tensor(-0.6492, dtype=torch.float64)
tensor(-0.6494, dtype=torch.float64)
tensor(-0.6495, dtype=torch.float64)
t

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.2999093532562256' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.3000396192073822' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/caoanjie/Desktop/projects/looking_time_models/03_pyGRANCH_multi/granch_utils/init_model_tensor.py:302: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.29998478293418884' has dtype incompatible with int64, please explicitly cast to a compatible dt

tensor(-1.1465, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1466, dtype=torch.float64)
tensor(-1.1466, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1466, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1466, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1466, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
tensor(-1.1464, dtype=torch.float64)
tensor(-1.1465, dtype=torch.float64)
t

KeyboardInterrupt: 

In [ ]:
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px

df = main_res

# Get unique categories
id = df['id'].unique()
# Create a subplot grid
fig = sp.make_subplots(rows=len(id), cols=1, subplot_titles=["1", "2", "3", "4", "5"])

# Populate each subplot
for i, idx in enumerate(id):
    subset = df[df['id'] == idx]
    print(subset)
    single_fig = px.line(df, x='t', y=['kl1', 'kl2'], labels={'kl1': 'kl1', 'kl2': 'kl2'}, markers=True)
    single_fig.update_layout(
    xaxis_title='t',
    yaxis_title='Value',
    legend_title='Legend',
    )


    for trace in single_fig.data:
        fig.add_trace(trace, row=i + 1, col=1)

# Update subplot layout (optional)
fig.update_layout(
    title='Faceted Line Plots',
    showlegend=True,
    height=600,
    width=400,
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px



df = res.iloc[:50]

# Create a line plot using Plotly Express
fig = px.line(df, x='t', y=['kl1', 'kl2'], labels={'kl1': 'kl1', 'kl2': 'kl2'}, markers=True)

# Customize the layout (optional)
fig.update_layout(
    title='Line Plot with Plotly',
    xaxis_title='t',
    yaxis_title='Value',
    legend_title='Legend',
)

# Show the plot
fig.show()


# Toy Example Test 

In [179]:
importlib.reload(init_model_tensor)
importlib.reload(lesioned_sim)
importlib.reload(init_params_tensor)
importlib.reload(compute_prob_tensor)
importlib.reload(main_sim_tensor)
importlib.reload(proxy_sim)

test_stimuli = init_model_tensor.granch_stimuli(3, "BBBBBD")
test_stimuli.add_toy_example(0.3, 0.7)

tensor_model =  init_model_tensor.granch_model(PRIOR_INFO['max_observation'], tensor_stimuli[0])

params = init_params_tensor.granch_params(
                grid_mu =  BATCH_GRID_INFO["grid_mus"][0].to(device),
                grid_sigma = BATCH_GRID_INFO["grid_sigmas"][0].to(device),
                grid_y = BATCH_GRID_INFO["grid_ys"][0].to(device),
                grid_epsilon = BATCH_GRID_INFO["grid_epsilons"][0].to(device),
                hypothetical_obs_grid_n = PRIOR_INFO["hypothetical_obs_grid_n"], 
                mu_prior = PRIOR_INFO["mu_prior"],
                V_prior = PRIOR_INFO["V_prior"], 
                alpha_prior = PRIOR_INFO["alpha_prior"], 
                beta_prior = PRIOR_INFO["beta_prior"],
                epsilon  = PRIOR_INFO["epsilon"], 
                mu_epsilon = PRIOR_INFO["mu_epsilon"], 
                sd_epsilon = PRIOR_INFO["sd_epsilon"], 
                world_EIGs = 3,
                #world_EIGs = PRIOR_INFO["world_EIGs"],
                #max_observation = PRIOR_INFO["max_observation"], 
                max_observation = 500,
                forced_exposure_max= np.nan, 
                linking_hypothesis= "surprisal")
        
            # add the various different cached bits
params.add_meshed_grid()
params.add_lp_mu_sigma()
params.add_y_given_mu_sigma()
params.add_lp_epsilon()
params.add_priors()

pd.set_option('display.max_rows', None)

#res = main_sim_tensor.granch_main_simulation(params, tensor_model, test_stimuli)
res = proxy_sim.granch_proxy_sim(params, tensor_model, test_stimuli)
#res.behavior

res.behavior

NameError: name 'tensor_stimuli' is not defined

In [ ]:
res.behavior

In [303]:
import torch.nn.functional as F

new_post = torch.tensor([[[[ 4.1338e-01, 9.9999e-321],
          [ 3.6698e-02, 9.9999e-321]],

         [[ 4.8830e-01, 9.9999e-321],
          [ 6.1620e-02, 9.9999e-321]]],


        [[[ 4.1338e-01, 9.9999e-321],
          [ 3.6698e-02, 9.9999e-321]],

         [[ 4.8830e-01, 9.9999e-321],
          [ 6.1619e-02, 9.9999e-321]]],


        [[[ 4.1338e-01, 9.9999e-321],
          [ 3.6698e-02, 9.9999e-321]],

         [[ 4.8830e-01, 9.9999e-321],
          [ 6.1620e-02, 9.9999e-321]]]], dtype=torch.float64)

paded_prev_post = torch.tensor([[[[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]],

         [[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]]],


        [[[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]],

         [[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]]],


        [[[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]],

         [[1.0000e+00, 1.0000e-31],
          [1.3534e-01, 1.0000e-31]]]])

#print(new_post/paded_prev_post)
#print(torch.log(new_post/paded_prev_post))

KL1 = torch.sum(torch.mul(new_post,
                         torch.log(new_post/paded_prev_post)), dim = (1, 2, 3))

KL2 = F.kl_div(torch.log(paded_prev_post),new_post, reduction = "none").sum(dim = (1, 2, 3))

print(KL1)
print(KL2)

print(new_post.sum())
print(paded_prev_post.sum(dim = (1, 2, 3)))
print(new_post < paded_prev_post)

print(torch.log(new_post/paded_prev_post))

tensor([-0.8116, -0.8116, -0.8116], dtype=torch.float64)
tensor([-0.8116, -0.8116, -0.8116], dtype=torch.float64)
tensor(3.0000, dtype=torch.float64)
tensor([2.2707, 2.2707, 2.2707])
tensor([[[[True, True],
          [True, True]],

         [[True, True],
          [True, True]]],


        [[[True, True],
          [True, True]],

         [[True, True],
          [True, True]]],


        [[[True, True],
          [True, True]],

         [[True, True],
          [True, True]]]])
tensor([[[[  -0.8834, -665.4471],
          [  -1.3051, -665.4471]],

         [[  -0.7168, -665.4471],
          [  -0.7868, -665.4471]]],


        [[[  -0.8834, -665.4471],
          [  -1.3051, -665.4471]],

         [[  -0.7168, -665.4471],
          [  -0.7868, -665.4471]]],


        [[[  -0.8834, -665.4471],
          [  -1.3051, -665.4471]],

         [[  -0.7168, -665.4471],
          [  -0.7868, -665.4471]]]], dtype=torch.float64)


# Test KL overflow

In [241]:

A = torch.tensor([9.9999e-321,  2.4348e-01])
B = torch.tensor([1.0000e-31, 3.6788e-01])
        
A_cast_first = torch.tensor([9.9999e-321,  2.4348e-01], dtype=torch.float64)
B_cast_first = torch.tensor([1.0000e-31, 3.6788e-01], dtype=torch.float64)  

A_cast_later = A.to(torch.float64)
B_cast_later = B.to(torch.float64)


mask = A_cast_later < 1e-10
A_cast_later[mask] = 1e-10 
mask = B_cast_later < 1e-10
B_cast_later[mask] = 1e-10


KL1 = torch.sum(torch.mul(A,torch.log(A/B)))
KL2 = torch.sum(torch.mul(A_cast_later,torch.log(A_cast_later/B_cast_later)))
KL3 =  torch.sum(torch.mul(A_cast_first,torch.log(A_cast_first/B_cast_first)))

print(KL1)
print(KL2)
print(KL3)

tensor(nan)
tensor(-0.1005, dtype=torch.float64)
tensor(-0.1005, dtype=torch.float64)


In [168]:
A = torch.tensor([9.9999e-321])
B = A.to(torch.float64)
print(A)
print(B)
print(B.dtype)

tensor([0.])
tensor([0.], dtype=torch.float64)
torch.float64


In [124]:
pred = torch.tensor([[0.2, 0.8]])
target = torch.tensor([[0.1, 0.9]])

kl2_a = F.kl_div(pred, target, reduction='sum', log_target=False)

kl2_b = F.kl_div(target, pred, reduction='sum', log_target=True)

KL1 = torch.sum(torch.mul(new_post, torch.log(new_post/paded_prev_post)), dim = 0)

print(kl2_a)
print(kl2_b)

print(torch.sum(torch.exp(target) * (target - pred)))


tensor(-1.0651)
tensor(-0.1004)
tensor(0.1354)
